# Case Study: Educational Multilevel Analysis

**Domain:** Education Research  
**Design:** Nested/Hierarchical (Students within Classes within Schools)  
**Analysis:** 3-Level Mixed Effects Models

## Background

An education researcher studies math achievement across a school district. Students are nested within classes, which are nested within schools.

**Research questions:**
1. How much variability exists at each level (student, class, school)?
2. What is the effect of class size on achievement?
3. Do schools with higher SES perform better?
4. What proportion of variance is contextual vs individual?

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from aurora.models.gamm import fit_gamm

sns.set_style('whitegrid')
np.random.seed(789)

## 1. Data Generation

**Hierarchical structure:**
- 20 schools
- 5 classes per school (100 classes total)
- 20 students per class (2000 students total)

In [ ]:
# Structure
n_schools = 20
n_classes_per_school = 5
n_students_per_class = 20
n_classes = n_schools * n_classes_per_school
n_students = n_classes * n_students_per_class

# True parameters
population_mean = 70.0
school_sd = 8.0  # school-level variance
class_sd = 5.0   # class-level variance (within schools)
student_sd = 12.0  # student-level variance

# Contextual effects
class_size_effect = -0.3  # smaller classes better
school_ses_effect = 10.0  # high SES schools higher achievement

# Generate school-level data
schools = pd.DataFrame({
    'school_id': range(n_schools),
    'school_ses': np.random.choice([0, 1], n_schools),  # 0=low, 1=high
    'school_effect': np.random.randn(n_schools) * school_sd
})

# Generate class-level data
classes = []
class_id = 0
for school_id in range(n_schools):
    for _ in range(n_classes_per_school):
        classes.append({
            'class_id': class_id,
            'school_id': school_id,
            'class_size': np.random.randint(15, 30),
            'class_effect': np.random.randn() * class_sd
        })
        class_id += 1
classes = pd.DataFrame(classes)

# Merge school info
classes = classes.merge(schools, on='school_id')

# Generate student-level data
data = []
student_id = 0
for _, cls in classes.iterrows():
    for _ in range(n_students_per_class):
        score = (population_mean +
                cls['school_effect'] +
                cls['class_effect'] +
                school_ses_effect * cls['school_ses'] +
                class_size_effect * (cls['class_size'] - 22) +  # centered
                np.random.randn() * student_sd)
        
        data.append({
            'student_id': student_id,
            'class_id': cls['class_id'],
            'school_id': cls['school_id'],
            'school_ses': cls['school_ses'],
            'class_size': cls['class_size'],
            'math_score': np.clip(score, 0, 100)
        })
        student_id += 1

df = pd.DataFrame(data)
print(f"Dataset: {n_students} students in {n_classes} classes in {n_schools} schools")

## 2. Descriptive Statistics

In [ ]:
# Summary by level
print("\nSchool-level summary:")
school_summary = df.groupby('school_id')['math_score'].agg(['mean', 'std', 'count'])
print(f"Mean scores range: {school_summary['mean'].min():.1f} to {school_summary['mean'].max():.1f}")
print(f"Between-school SD: {school_summary['mean'].std():.2f}")

print("\nClass-level summary:")
class_summary = df.groupby('class_id')['math_score'].agg(['mean', 'std'])
print(f"Mean scores range: {class_summary['mean'].min():.1f} to {class_summary['mean'].max():.1f}")
print(f"Between-class SD: {class_summary['mean'].std():.2f}")

print("\nSES effect (descriptive):")
ses_summary = df.groupby('school_ses')['math_score'].agg(['mean', 'std', 'count'])
print(ses_summary)

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# School distributions
for school_id in range(min(10, n_schools)):
    school_data = df[df['school_id'] == school_id]['math_score']
    axes[0].violinplot([school_data], positions=[school_id], widths=0.7,
                      showmeans=True, showmedians=False)
axes[0].set_xlabel('School ID')
axes[0].set_ylabel('Math Score')
axes[0].set_title('Score Distribution by School (first 10)')
axes[0].grid(alpha=0.3, axis='y')

# SES effect
df.boxplot(column='math_score', by='school_ses', ax=axes[1])
axes[1].set_xlabel('School SES (0=Low, 1=High)')
axes[1].set_ylabel('Math Score')
axes[1].set_title('Achievement by School SES')
plt.sca(axes[1])
plt.xticks([1, 2], ['Low', 'High'])

# Class size effect
class_means = df.groupby('class_id').agg({'math_score': 'mean', 'class_size': 'first'})
axes[2].scatter(class_means['class_size'], class_means['math_score'], 
               alpha=0.6, s=50, edgecolor='black')
z = np.polyfit(class_means['class_size'], class_means['math_score'], 1)
p = np.poly1d(z)
axes[2].plot(class_means['class_size'].sort_values(), 
            p(class_means['class_size'].sort_values()),
            'r--', linewidth=2, label=f'Slope={z[0]:.2f}')
axes[2].set_xlabel('Class Size')
axes[2].set_ylabel('Class Mean Score')
axes[2].set_title('Achievement by Class Size')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Multilevel Model

**Model**: score ~ ses + class_size + (1 | school) + (1 | class)

In [ ]:
# Center class size
df['class_size_c'] = df['class_size'] - df['class_size'].mean()

# Fit 3-level model
result = fit_gamm(
    formula='math_score ~ school_ses + class_size_c + (1 | school_id) + (1 | class_id)',
    data=df,
    family='gaussian',
    covariance='identity'
)

print("\n" + "="*70)
print("Multilevel Model Results: Educational Achievement")
print("="*70)
print(f"\nConverged: {result.converged} (iterations: {result.n_iterations})")

# Fixed effects
print("\nFixed Effects:")
print(f"  Intercept: {result.beta_parametric[0]:.2f}")
print(f"  School SES: {result.beta_parametric[1]:.2f} (true: {school_ses_effect:.1f})")
print(f"  Class Size: {result.beta_parametric[2]:.2f} per student (true: {class_size_effect:.2f})")

# Random effects
school_sd_est = np.sqrt(result.variance_components[0][0, 0])
class_sd_est = np.sqrt(result.variance_components[1][0, 0])
student_sd_est = np.sqrt(result.residual_variance)

print("\nVariance Components:")
print(f"  School SD: {school_sd_est:.2f} (true: {school_sd:.1f})")
print(f"  Class SD: {class_sd_est:.2f} (true: {class_sd:.1f})")
print(f"  Student SD: {student_sd_est:.2f} (true: {student_sd:.1f})")

# Intraclass correlations
total_var = (result.variance_components[0][0,0] + 
            result.variance_components[1][0,0] + 
            result.residual_variance)

icc_school = result.variance_components[0][0,0] / total_var
icc_class = result.variance_components[1][0,0] / total_var
icc_student = result.residual_variance / total_var

print("\nVariance Partition (ICC):")
print(f"  School level: {icc_school:.3f} ({icc_school*100:.1f}%)")
print(f"  Class level: {icc_class:.3f} ({icc_class*100:.1f}%)")
print(f"  Student level: {icc_student:.3f} ({icc_student*100:.1f}%)")
print(f"\nContextual variance (school + class): {(icc_school + icc_class)*100:.1f}%")

## 4. Variance Decomposition Visualization

In [ ]:
# Pie chart of variance components
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Variance percentages
components = ['School', 'Class', 'Student']
percentages = [icc_school*100, icc_class*100, icc_student*100]
colors = ['#ff9999', '#66b3ff', '#99ff99']

axes[0].pie(percentages, labels=components, autopct='%1.1f%%',
           colors=colors, startangle=90, textprops={'fontsize': 12, 'weight': 'bold'})
axes[0].set_title('Variance Decomposition', fontsize=14, fontweight='bold')

# Bar chart
axes[1].bar(components, percentages, color=colors, edgecolor='black', linewidth=2)
axes[1].set_ylabel('Percent of Total Variance', fontsize=12, fontweight='bold')
axes[1].set_title('Variance at Each Level', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3, axis='y')
for i, pct in enumerate(percentages):
    axes[1].text(i, pct + 2, f'{pct:.1f}%', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Policy Implications

In [ ]:
print("\n" + "="*70)
print("Policy-Relevant Findings")
print("="*70)

# Class size effect
class_size_10_reduction = result.beta_parametric[2] * (-10)
print(f"\n1. CLASS SIZE REDUCTION:")
print(f"   Reducing class size by 10 students:")
print(f"   Expected gain: {class_size_10_reduction:.2f} points")
print(f"   Effect size (in student SDs): {class_size_10_reduction / student_sd_est:.3f}")

# SES effect
ses_effect = result.beta_parametric[1]
print(f"\n2. SOCIOECONOMIC DISPARITIES:")
print(f"   High-SES vs Low-SES schools differ by: {ses_effect:.2f} points")
print(f"   This is {ses_effect / student_sd_est:.2f} student-level standard deviations")
print(f"   Represents {ses_effect / population_mean * 100:.1f}% of mean achievement")

# Clustering
print(f"\n3. CONTEXTUAL EFFECTS:")
print(f"   {(icc_school + icc_class)*100:.1f}% of variance is between schools/classes")
print(f"   This indicates {'strong' if (icc_school + icc_class) > 0.20 else 'moderate'} clustering")
print(f"   Interventions should target {'schools/classes' if icc_school > 0.10 else 'individual students'}")

# Design effects
design_effect = 1 + (n_students_per_class - 1) * (icc_school + icc_class)
print(f"\n4. STATISTICAL CONSIDERATIONS:")
print(f"   Design effect: {design_effect:.2f}")
print(f"   Effective sample size: {n_students / design_effect:.0f} (vs {n_students} students)")
print(f"   Ignoring clustering would lead to {np.sqrt(design_effect):.2f}× underestimated SEs")

## 6. Conclusions

### Key Findings

1. **Variance Decomposition**:
   - Most variance (~70%) is between individual students
   - School-level factors explain ~15-20% of variance
   - Class-level factors explain ~10-15% of variance

2. **Socioeconomic Disparities**:
   - High-SES schools score ~10 points higher on average
   - This represents a meaningful achievement gap
   - Suggests need for equity-focused interventions

3. **Class Size Effects**:
   - Smaller classes associated with higher achievement
   - Effect size is modest but consistent
   - Cost-benefit analysis needed for policy decisions

4. **Design Implications**:
   - Students within schools/classes are not independent
   - Ignoring clustering leads to anti-conservative tests
   - Multilevel models are essential for valid inference

### Why Multilevel Modeling Matters

**For Research**:
- Correct standard errors and p-values
- Separates individual and contextual effects
- Identifies appropriate intervention levels

**For Policy**:
- Quantifies school/class quality differences
- Informs resource allocation decisions
- Evaluates equity and access issues

### References

- Raudenbush & Bryk (2002). *Hierarchical Linear Models*
- Snijders & Bosker (2011). *Multilevel Analysis*
- Hox et al. (2017). *Multilevel Analysis: Techniques and Applications*